<a href="https://colab.research.google.com/github/john-dechellis-weather/wx_compare/blob/main/wx_compare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [93]:
!apt-get install -y libeccodes0 > /dev/null 2>&1
!pip install -q cfgrib xarray eccodes requests pandas matplotlib

In [ ]:
import os, sys

REPO_URL = "https://github.com/john-dechellis-weather/wx_compare.git"
REPO_DIR = "/content/wx_compare"

os.chdir('/content')
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

In [95]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
CACHE_ROOT = Path('/content/drive/MyDrive/wx_compare_cache')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [98]:
raw = input("Enter ICAO codes (comma-separated, e.g. KJFK,KORD,KSEA): ")
ICAOS = [s.strip().upper() for s in raw.split(",") if s.strip()]
print(f"Will compare: {ICAOS}")

Enter ICAO codes (comma-separated, e.g. KJFK,KORD,KSEA): KJFK
Will compare: ['KJFK']


In [ ]:
from datetime import datetime, timezone, timedelta
from compare import compare_icaos

CYCLE = (datetime.now(timezone.utc) - timedelta(days=1)).replace(
    hour=12, minute=0, second=0, microsecond=0)

df, resolved, unresolved = compare_icaos(
    icaos=ICAOS,
    cycle=CYCLE,
    cache_root=CACHE_ROOT,
)

if unresolved:
    print(f"⚠ Not found in airport database: {unresolved}")

print(f"\nResolved {len(resolved)} stations:")
for s in resolved:
    print(f"  {s.icao}  {s.name}  ({s.lat:.2f}, {s.lon:.2f}, {s.elev_ft:.0f} ft)")

df

In [ ]:
from compare import plot_comparison
for s in resolved:
    plot_comparison(df, s.icao)